[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microscopy-processing/N2N-SPRVS/blob/main/N2N-SPRVS.ipynb)

# Phantom SNR=01 24nm

In [ ]:
import mrcfile

In [ ]:
import numpy as np
sliice = np.s_[:, :]

In [ ]:
# %pip install tqdm ipywidgets
from tqdm.notebook import tqdm

In [ ]:
# %pip install matplotlib
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

In [ ]:
import os

In [ ]:
from collections import namedtuple

In [ ]:
Args = namedtuple("args", ["X", "Y_REO", "Y_SPRVS", "Z"])
args = Args("/home/vruiz/Tomograms/tomograms/phantom/phantom_snr01_24nm.mrc",
            "/home/vruiz/REO/phantom/snr01_24nm/REO/denoised_vol/phantom_snr01_24nm.mrc",
            "./denoised_vol_SPRVS/phantom_snr01_24nm.mrc",
            "/home/vruiz/Tomograms/tomograms/phantom/phantom_24nm.mrc"
           )

In [ ]:
X = mrcfile.open(args.X).data

In [ ]:
Y_SPRVS = mrcfile.open(args.Y_SPRVS).data

In [ ]:
Y_REO = mrcfile.open(args.Y_REO).data

In [ ]:
Z = mrcfile.open(args.Z).data

In [ ]:
!pwd

In [ ]:
X.shape

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 4, figsize=(20, 20))

_ = axes[0].imshow(Z[slice_idx][sliice], cmap='gray', origin='lower')
axes[0].set_title(f'Clean_Z={slice_idx}')
axes[0].grid(False)

_ = axes[1].imshow(X[slice_idx][sliice], cmap='gray', origin='lower')
axes[1].set_title(f'Noisy_Z={slice_idx}')
axes[1].grid(False)

_ = axes[2].imshow(Y_SPRVS[slice_idx][sliice], cmap='gray', origin='lower')
axes[2].set_title(f'Denoised_SPRVS Z={slice_idx}')
axes[2].grid(False)

_ = axes[3].imshow(X[slice_idx][sliice]- Y_SPRVS[slice_idx][sliice] + 128, cmap='gray', origin='lower')
axes[3].set_title(f'Noisy - Denoised_SPRVS Z={slice_idx}')
axes[3].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# %pip install "self_fourier_shell_correlation @ git+https://github.com/vicente-gonzalez-ruiz/self_fourier_shell_correlation"
# %pip install "shuffling @ git+https://github.com/vicente-gonzalez-ruiz/shuffling"
# %pip install "motion_estimation @ git+https://github.com/vicente-gonzalez-ruiz/motion_estimation"
from self_fourier_shell_correlation import fsc_utils as fsc

In [ ]:
# %pip show self_fourier_shell_correlation

In [ ]:
# %pip install  --force-reinstall --no-cache-dir "shuffling @ git+https://github.com/vicente-gonzalez-ruiz/shuffling"

In [ ]:
list_fsc_values__X = []
list_fsc_values__Y_SPRVS = []
list_fsc_values__Y_REO = []
list_fsc_values__Z = []
for i in range(X.shape[0]-2):
    print(i, '/', X.shape[0])
    spatial_freqs, fsc_values__X = fsc.get_SFRC_curve__subsampled_chessboard(X[i])
    spatial_freqs, fsc_values__Y_SPRVS = fsc.get_SFRC_curve__subsampled_chessboard(Y_SPRVS[i])
    spatial_freqs, fsc_values__Y_REO = fsc.get_SFRC_curve__subsampled_chessboard(Y_REO[i])
    spatial_freqs, fsc_values__Z = fsc.get_SFRC_curve__subsampled_chessboard(Z[i])
    list_fsc_values__X.append(fsc_values__X)
    list_fsc_values__Y_SPRVS.append(fsc_values__Y_SPRVS)
    list_fsc_values__Y_REO.append(fsc_values__Y_REO)
    list_fsc_values__Z.append(fsc_values__Z)

In [ ]:
avg_fsc_values__X = np.mean(list_fsc_values__X, axis=0)
avg_fsc_values__Y_SPRVS = np.mean(list_fsc_values__Y_SPRVS, axis=0)
avg_fsc_values__Y_REO = np.mean(list_fsc_values__Y_REO, axis=0)
avg_fsc_values__Z = np.mean(list_fsc_values__Z, axis=0)

In [ ]:
plt.title(f"{Path.cwd().parts[-2:]}")
plt.xlabel("Normalized Spatial Frequency (cycles/pixel)")
plt.ylabel("Average Self Fourier Ring Correlation")
plt.plot(spatial_freqs, avg_fsc_values__X, label="Noisy", color="blue")
plt.plot(spatial_freqs, avg_fsc_values__Y_REO, label="N2N-REO", color="cyan")
plt.plot(spatial_freqs, avg_fsc_values__Y_SPRVS, label="N2N-SPRVS", color="green")
plt.plot(spatial_freqs, avg_fsc_values__Z, label="Clean", color="red")
plt.legend(loc='lower left')

In [ ]:
import scipy.stats

In [ ]:
def PCC(A, B):
    return scipy.stats.pearsonr(A.flatten(), B.flatten())[0]

In [ ]:
print(f"PCC(Z, Y_SPRVS)={PCC(Z, Y_SPRVS):.3f}")

In [ ]:
print(f"PCC(X, Y_SPRVS)={PCC(X, Y_SPRVS):.3f}")

In [ ]:
print(f"PCC(X, Z)={PCC(X, Z):.3f}")

In [ ]:
#%pip install scikit-image
#%conda install -c conda-forge scikit-image
import skimage.metrics

In [ ]:
def PSNR(A, B):
    return skimage.metrics.peak_signal_noise_ratio(A, B, data_range=A.max() - A.min())

In [ ]:
print(f"PSNR(Z, Y_SPRVS)={PSNR(Z, Y_SPRVS):.3f}")

In [ ]:
print(f"PSNR(X, Y_SPRVS)={PSNR(X, Y_SPRVS):.3f}")

In [ ]:
print(f"PSNR(X, Z)={PSNR(X, Z):.3f}")

In [ ]:
def SSIM(A, B):
    return skimage.metrics.structural_similarity(A, B, data_range=A.max() - A.min())

In [ ]:
print(f"SSIM(Z, Y_SPRVS)={SSIM(Z, Y_SPRVS):.3f}")

In [ ]:
print(f"SSIM(X, Y_SPRVS)={SSIM(X, Y_SPRVS):.3f}")

In [ ]:
print(f"SSIM(X, Z)={SSIM(X, Z):.3f}")